# Parte di Francesco — Recupero e preparazione dei dati della pratica

**Traccia 5 — Assistente per la gestione di una pratica assicurativa (Gruppo 6)**

In questo notebook preparo le funzioni che **leggono i dati** del progetto e **raccolgono in un unico dizionario** tutte le informazioni che servono agli altri componenti dell'assistente (verifica copertura, controllo documenti, decisione automatica o perito).

> **Cosa NON fa questo notebook:** non approva e non respinge nessuna pratica. Le regole deterministiche (es. "il danno supera la soglia → serve un perito") saranno applicate dal codice degli altri componenti, come chiede la traccia. Qui ci occupiamo solo di **recuperare i dati in modo affidabile**.

### Come eseguire il notebook
1. Apri il notebook dalla cartella principale del progetto, `EsameGENAI-main` (la stessa che contiene la cartella `data`). Jupyter e VS Code usano come cartella di lavoro quella del notebook, quindi basta aprire `Francesco.ipynb` da lì.
2. Esegui **"Riavvia ed esegui tutto"** (*Restart & Run All*): ogni cella usa solo ciò che è definito nelle celle precedenti.
3. Serve solo Python con la libreria standard (`json`, `pathlib`, `tempfile`): nessuna installazione aggiuntiva, nessun modello LLM.

## Materiali delle lezioni consultati

Ho letto questi materiali del corso e ne ho ripreso solo le idee adatte al nostro compito:

| Materiale | Cosa ho ripreso |
|---|---|
| II anno — `Giorno 3/Lab/02_Agente_con_Tools_Ollama.ipynb` (tool `get_employee_budget`) | Cartella dati con `Path("data")`, lettura del JSON con `json.load`, pulizia dell'identificativo con `.strip().upper()`, risposta del tool come dizionario con un campo `status` (`"OK"` / `"NOT_FOUND"`) e un messaggio leggibile. |
| II anno — `Giorno 4/Lab/02_LangGraph_MVP.ipynb` | Caricamento dei JSON con `with open(...) as f: json.load(f)`; tool che restituisce un messaggio chiaro quando un elemento non esiste (`EMPLOYEES_DB.get(employee_id)` → "non trovato nel sistema"). |
| II anno — `Giorno 2/Lab/01_MCP/simple_mcp_server.py` | L'idea di distinguere due casi diversi: **elemento non trovato** (`TOOL_NOT_FOUND`, `"status": "error"`) ed **errore durante l'esecuzione** (`TOOL_EXECUTION_ERROR`, intercettato con `try/except`). |
| I anno — `Lezione 31 settembre/03_introduzione_ai_agenti.ipynb` (tool `leggi_file`) | Controllare che il file esista prima di aprirlo, aprire con `encoding="utf-8"`, docstring con la descrizione di argomenti e valore restituito. |
| I anno — `Lezione 18 settembre/Soluzioni/06_Esercizi_Python_Strutture_SOLUZIONI.ipynb` | Funzione di ricerca che restituisce `None` se l'elemento non c'è (`cerca_prodotto`), cicli `for` su dizionari, `try/except` con **tipi di errore specifici** (`ValueError`, ...) invece di un `except` generico. |

**Cosa ho semplificato e perché**
- Nelle lezioni le funzioni sono decorate con `@tool` (LangChain) o `@mcp.tool()`, e il server MCP usa una classe `FastMCP`. Qui **non** li uso: richiederebbero librerie esterne e una classe, mentre il mio compito è solo preparare i dati. Le funzioni restano normali funzioni Python: chi costruisce l'agente potrà trasformarle in tool aggiungendo `@tool` sopra `estrai_variabili_pratica`.
- Nelle lezioni i dipendenti sono in un dizionario indicizzato per ID (`employees[employee_id]`); i nostri file invece sono **liste** di record, quindi la ricerca usa un semplice ciclo `for` con un `if`.
- `search_product_catalog` usa list comprehension e `any(...)`: qui preferisco cicli espliciti, più facili da leggere e da spiegare.

## I dati del progetto e i loro collegamenti

I dati sono file **JSON** nella cartella `data`.

> **JSON** (*JavaScript Object Notation*) è un formato di testo per salvare dati strutturati. Le parentesi quadre `[ ]` indicano una **lista** (una sequenza ordinata di elementi), le parentesi graffe `{ }` un **oggetto**, che in Python diventa un **dizionario**: un insieme di coppie *chiave → valore*, ad esempio `{"claim_id": "CL001"}` dove `"claim_id"` è la chiave e `"CL001"` il valore.

| File | Contenuto | Collegamento |
|---|---|---|
| `claims.json` | sinistri (le richieste di risarcimento) | ogni sinistro ha un `policy_id` → la polizza |
| `policies.json` | polizze | ogni polizza ha un `customer_id` → il cliente |
| `customers.json` | clienti (assicurati) | — |
| `requirements.json` | documenti richiesti e soglia per tipo di sinistro | si collega tramite `loss_type` |
| `README.json` | descrizione generale del dataset | non serve per i controlli |

Il percorso per raccogliere i dati di una pratica è quindi:

```
sinistro (claim_id) ──policy_id──► polizza ──customer_id──► cliente
        └──────────loss_type──────────► requisiti
```

## Significato delle chiavi del dizionario finale

Le chiavi restano **quelle originali dei JSON** (in inglese) così gli altri componenti possono usarle senza conversioni. Ecco cosa significano:

| Chiave | Da quale file | Significato | Tipo |
|---|---|---|---|
| `claim_id` | claims | codice del sinistro (es. `"CL001"`) | testo |
| `policy_id` | claims → policies | codice della polizza collegata | testo |
| `customer_id` | policies → customers | codice del cliente | testo |
| `name` | customers | nome e cognome del cliente | testo |
| `risk_tier` | customers | categoria di rischio del cliente (`"standard"`, `"premium"`) | testo |
| `product` | policies | prodotto assicurativo (`"Auto"`, `"Home"`) | testo |
| `coverage` | policies | tipo di evento coperto dalla polizza (es. `"collision"`) | testo |
| `active` | policies | `True` se la polizza è attiva, `False` se non lo è | booleano |
| `deductible` | policies | franchigia: la parte di danno che resta a carico del cliente | numero |
| `max_claim` | policies | massimale: il risarcimento massimo previsto dalla polizza | numero |
| `loss_type` | claims | tipo di sinistro (`"collision"` = urto/incidente, `"water_damage"` = danno da acqua, `"theft"` = furto) | testo |
| `estimated_damage` | claims | danno stimato in euro | numero |
| `documents` | claims | documenti già consegnati (es. `["photo", "police_report"]`) | lista |
| `required_documents` | requirements | documenti obbligatori per quel tipo di sinistro | lista |
| `automatic_review_max` | requirements | importo massimo del danno per cui la pratica può essere gestita in automatico | numero |

Oltre a questi dati, il dizionario contiene tre chiavi di **controllo**, scritte in italiano perché sono aggiunte da noi:

| Chiave | Significato |
|---|---|
| `status` | esito del recupero: `"OK"`, `"INCOMPLETO"`, `"NON_TROVATO"` oppure `"ERRORE_LETTURA"` |
| `messaggio` | spiegazione in italiano dell'esito |
| `campi_mancanti` | lista dei campi che non erano presenti nei file (vuota se tutto è stato trovato) |

> Nota: `photo` = fotografie del danno, `police_report` = denuncia/verbale delle autorità, `repair_invoice` = fattura di riparazione.

## Preparazione: librerie e cartella dei dati

Importiamo due librerie della **libreria standard** di Python (già incluse, non serve installarle):
- `json` trasforma il testo JSON in liste e dizionari Python;
- `pathlib` gestisce i percorsi dei file in modo che funzionino sia su Windows sia su macOS/Linux.

`CARTELLA_DATI` è un **percorso relativo**: `Path("data")` significa "la cartella `data` dentro la cartella da cui ho avviato il notebook". Per questo il notebook va aperto dalla cartella principale del progetto. La cella controlla subito che la cartella esista, così un problema di percorso si vede all'inizio.

In [1]:
import json
from pathlib import Path

# Cartella dei dati, relativa alla cartella principale del progetto (EsameGENAI-main)
CARTELLA_DATI = Path("data")

if CARTELLA_DATI.exists():
    print("Cartella dati trovata:", CARTELLA_DATI.resolve())
else:
    print("ATTENZIONE: cartella 'data' non trovata in", Path.cwd().resolve())
    print("Avvia il notebook dalla cartella EsameGENAI-main.")

Cartella dati trovata: C:\Users\A847apulia\Desktop\EsameGENAI-main\data


## 1. `carica_dati(nome_file)` — leggere un file JSON

**A cosa serve.** Apre un file della cartella `data` e trasforma il suo contenuto JSON in dati Python (liste e dizionari).

**Cosa riceve.** Un **parametro** è un valore che passiamo alla funzione quando la chiamiamo, scritto tra le parentesi. Qui i parametri sono:
- `nome_file`: il nome del file, ad esempio `"claims.json"`;
- `cartella` (facoltativo): la cartella dove cercarlo. Se non lo indichiamo vale `CARTELLA_DATI`. Serve solo per le prove finali, dove useremo una cartella di test.

**Cosa restituisce.** L'istruzione `return` è il modo in cui una funzione "consegna" il risultato a chi l'ha chiamata (diversamente da `print`, che lo mostra soltanto a schermo). Questa funzione restituisce i dati letti: per i nostri file, una lista di dizionari.

**Come funziona.**
1. Costruisce il percorso completo, ad esempio `data/claims.json`.
2. Se il file non esiste, **solleva un errore** (`raise`) con un messaggio chiaro. Sollevare un errore significa interrompere la funzione e avvisare chi l'ha chiamata che qualcosa è andato storto.
3. Apre il file e lo legge con `json.load`. Se il testo non è un JSON valido (ad esempio manca una parentesi), Python genera un `json.JSONDecodeError`: lo intercettiamo con `try/except` e lo trasformiamo in un `ValueError` con un messaggio comprensibile.

**Perché un errore e non `None`?** Le funzioni di ricerca useranno `None` per dire "non c'è". Un file mancante o rotto è un problema diverso, un **errore di lettura**: usando un errore distinguiamo i due casi.

**Esempio.** `carica_dati("customers.json")` restituisce
`[{"customer_id": "AS001", "name": "Pietro Sala", "risk_tier": "standard"}, ...]`.

In [2]:
def carica_dati(nome_file, cartella=CARTELLA_DATI):
    """Legge un file JSON e restituisce i dati che contiene.

    Parametri:
        nome_file: nome del file, ad esempio "claims.json".
        cartella: cartella in cui si trova il file (di default la cartella "data").

    Restituisce:
        I dati letti dal file (per i nostri file: una lista di dizionari).

    Errori:
        FileNotFoundError se il file non esiste,
        ValueError se il file non contiene un JSON valido.
    """
    percorso = Path(cartella) / nome_file

    # Primo controllo: il file deve esistere
    if not percorso.exists():
        raise FileNotFoundError(f"Il file '{percorso}' non esiste. Controlla di aver avviato il notebook dalla cartella del progetto.")

    # Secondo controllo: il contenuto deve essere un JSON valido
    try:
        with open(percorso, "r", encoding="utf-8") as file:
            dati = json.load(file)
    except json.JSONDecodeError as errore:
        raise ValueError(f"Il file '{percorso}' non contiene un JSON valido (riga {errore.lineno}): {errore.msg}")

    return dati

In [3]:
# Prova: leggiamo i clienti e guardiamo il primo elemento della lista
clienti_prova = carica_dati("customers.json")
print("Numero di clienti:", len(clienti_prova))
print("Primo cliente:", clienti_prova[0])

Numero di clienti: 3
Primo cliente: {'customer_id': 'AS001', 'name': 'Pietro Sala', 'risk_tier': 'standard'}


## 2. `cerca_sinistro(id_sinistro, sinistri)` — trovare un sinistro

**A cosa serve.** Cerca, dentro la lista dei sinistri, quello con il codice richiesto.

**Cosa riceve.**
- `id_sinistro`: il codice da cercare, ad esempio `"CL002"`;
- `sinistri`: la lista dei sinistri letta da `claims.json`.

**Cosa restituisce.** Il dizionario del sinistro trovato, oppure `None` se nessun sinistro ha quel codice. `None` in Python significa "nessun valore": è un modo semplice per dire "non trovato" senza causare errori.

**Come funziona.** Scorre la lista con un ciclo `for`, un elemento alla volta. Per ogni sinistro controlla con `if` se il valore della chiave `"claim_id"` è uguale al codice cercato. Appena lo trova lo restituisce con `return` (e la funzione termina). Se il ciclo finisce senza trovarlo, arriva all'ultima riga e restituisce `None`.

Uso `sinistro.get("claim_id")` invece di `sinistro["claim_id"]`: se un record non avesse quella chiave, `.get` restituisce `None` invece di bloccare il programma con un errore.

**Esempio.** `cerca_sinistro("CL002", sinistri)` → `{"claim_id": "CL002", "policy_id": "POL002", "loss_type": "water_damage", ...}`;
`cerca_sinistro("CL999", sinistri)` → `None`.

In [4]:
def cerca_sinistro(id_sinistro, sinistri):
    """Cerca un sinistro tramite il suo codice (claim_id).

    Restituisce il dizionario del sinistro, oppure None se non esiste.
    """
    for sinistro in sinistri:
        if sinistro.get("claim_id") == id_sinistro:
            return sinistro
    return None

## 3. `cerca_polizza(id_polizza, polizze)` — trovare la polizza del sinistro

**A cosa serve.** Trova la polizza collegata al sinistro. Il collegamento è il campo `policy_id` presente nel sinistro.

**Cosa riceve.**
- `id_polizza`: il codice della polizza, preso dal sinistro (es. `"POL001"`);
- `polizze`: la lista letta da `policies.json`.

**Cosa restituisce.** Il dizionario della polizza oppure `None` se non esiste.

**Come funziona.** Stessa logica di `cerca_sinistro`: un ciclo `for` sulla lista e un confronto con `if` sulla chiave `"policy_id"`.

**Esempio.** Il sinistro `CL004` ha `policy_id = "POL001"`; `cerca_polizza("POL001", polizze)` restituisce la polizza Auto di Pietro Sala. Più sinistri possono puntare alla stessa polizza (anche `CL001` usa `POL001`).

In [5]:
def cerca_polizza(id_polizza, polizze):
    """Cerca una polizza tramite il suo codice (policy_id).

    Restituisce il dizionario della polizza, oppure None se non esiste.
    """
    for polizza in polizze:
        if polizza.get("policy_id") == id_polizza:
            return polizza
    return None

## 4. `cerca_cliente(id_cliente, clienti)` — trovare il cliente della polizza

**A cosa serve.** Trova il cliente intestatario della polizza. Il collegamento è il campo `customer_id` presente nella polizza.

**Cosa riceve.**
- `id_cliente`: il codice del cliente, preso dalla polizza (es. `"AS002"`);
- `clienti`: la lista letta da `customers.json`.

**Cosa restituisce.** Il dizionario del cliente oppure `None`.

**Come funziona.** Ciclo `for` + confronto `if` sulla chiave `"customer_id"`.

**Esempio.** `cerca_cliente("AS002", clienti)` → `{"customer_id": "AS002", "name": "Chiara Riva", "risk_tier": "premium"}`.

In [6]:
def cerca_cliente(id_cliente, clienti):
    """Cerca un cliente tramite il suo codice (customer_id).

    Restituisce il dizionario del cliente, oppure None se non esiste.
    """
    for cliente in clienti:
        if cliente.get("customer_id") == id_cliente:
            return cliente
    return None

## 5. `cerca_requisiti(tipo_sinistro, requisiti)` — documenti richiesti e soglia

**A cosa serve.** Ogni tipo di sinistro ha regole diverse: quali documenti servono e fino a quale importo la pratica può essere gestita in automatico. Questa funzione recupera queste regole per il tipo di sinistro indicato.

**Cosa riceve.**
- `tipo_sinistro`: il valore `loss_type` del sinistro (es. `"theft"`);
- `requisiti`: la lista letta da `requirements.json`.

**Cosa restituisce.** Il dizionario con `required_documents` e `automatic_review_max`, oppure `None` se per quel tipo di sinistro non esistono regole.

**Come funziona.** Ciclo `for` + confronto `if` sulla chiave `"loss_type"`.

**Esempio.** `cerca_requisiti("theft", requisiti)` →
`{"loss_type": "theft", "required_documents": ["police_report", "photo"], "automatic_review_max": 1000}`.

In [7]:
def cerca_requisiti(tipo_sinistro, requisiti):
    """Cerca i requisiti (documenti richiesti e soglia) per un tipo di sinistro (loss_type).

    Restituisce il dizionario dei requisiti, oppure None se non esistono.
    """
    for requisito in requisiti:
        if requisito.get("loss_type") == tipo_sinistro:
            return requisito
    return None

In [8]:
# Prova delle quattro funzioni di ricerca, una alla volta
sinistri = carica_dati("claims.json")
polizze = carica_dati("policies.json")
clienti = carica_dati("customers.json")
requisiti = carica_dati("requirements.json")

sinistro = cerca_sinistro("CL002", sinistri)
print("Sinistro: ", sinistro)

polizza = cerca_polizza(sinistro["policy_id"], polizze)
print("Polizza:  ", polizza)

cliente = cerca_cliente(polizza["customer_id"], clienti)
print("Cliente:  ", cliente)

requisito = cerca_requisiti(sinistro["loss_type"], requisiti)
print("Requisiti:", requisito)

print("Sinistro inesistente:", cerca_sinistro("CL999", sinistri))

Sinistro:  {'claim_id': 'CL002', 'policy_id': 'POL002', 'loss_type': 'water_damage', 'estimated_damage': 7000, 'documents': ['photo']}
Polizza:   {'policy_id': 'POL002', 'customer_id': 'AS002', 'product': 'Home', 'coverage': 'water_damage', 'deductible': 500, 'max_claim': 10000, 'active': True}
Cliente:   {'customer_id': 'AS002', 'name': 'Chiara Riva', 'risk_tier': 'premium'}
Requisiti: {'loss_type': 'water_damage', 'required_documents': ['photo', 'repair_invoice'], 'automatic_review_max': 5000}
Sinistro inesistente: None


## 6. Funzione di supporto `copia_campi(...)` — copiare i valori senza inventarli

**A cosa serve.** Quando costruiamo il dizionario finale dobbiamo copiare molti campi dai record trovati. Se un campo manca nel file, **non** vogliamo inventare un valore (come `0`, `False` o `[]`): sarebbe pericoloso, perché ad esempio una franchigia `0` o una polizza `False` cambierebbero la decisione sulla pratica.

**Cosa riceve.**
- `origine`: il dizionario da cui copiare (sinistro, polizza, cliente o requisiti);
- `nomi_campi`: la lista dei nomi dei campi da copiare;
- `dati_pratica`: il dizionario finale dove scrivere i valori;
- `campi_mancanti`: la lista dove annotare i campi non trovati.

**Cosa restituisce.** Niente: modifica direttamente `dati_pratica` e `campi_mancanti`, che le vengono passati. In Python liste e dizionari passati a una funzione possono essere modificati dalla funzione stessa.

**Come funziona.** Per ogni nome di campo: se il campo esiste nell'origine, copia il valore **così com'è** (un numero resta numero, un booleano resta booleano, una lista resta lista). Se manca, scrive `None` (cioè "valore assente", non un valore inventato) e aggiunge il nome a `campi_mancanti`, così chi usa i dati sa esattamente cosa manca.

**Esempio.** Se una polizza non avesse `"deductible"`, il risultato conterrebbe `"deductible": None` e `campi_mancanti` conterrebbe `"deductible"`.

In [9]:
def copia_campi(origine, nomi_campi, dati_pratica, campi_mancanti):
    """Copia i campi indicati da 'origine' a 'dati_pratica' senza modificarne il tipo.

    Se un campo non esiste lo segna come None e lo aggiunge a 'campi_mancanti'.
    """
    for campo in nomi_campi:
        if campo in origine:
            dati_pratica[campo] = origine[campo]
        else:
            dati_pratica[campo] = None  # valore assente: non lo inventiamo
            campi_mancanti.append(campo)

## 7. `estrai_variabili_pratica(id_sinistro)` — la funzione principale

**A cosa serve.** È la funzione che useranno gli altri membri del gruppo. Dato il codice di un sinistro, raccoglie **tutte** le informazioni della pratica in un unico dizionario.

**Cosa riceve.**
- `id_sinistro`: il codice del sinistro, ad esempio `"CL001"`. Spazi iniziali/finali e minuscole vengono corretti (`" cl001 "` → `"CL001"`), come nel tool `get_employee_budget` delle lezioni: utile quando il codice arriva dal testo scritto dall'utente o estratto dal modello;
- `cartella` (facoltativo): la cartella dei dati, usata solo nelle prove.

**Cosa restituisce.** Sempre un dizionario con la chiave `status`, che può valere:

| `status` | Quando succede | Cosa contiene |
|---|---|---|
| `"OK"` | tutti i dati sono stati trovati | tutte le variabili della pratica |
| `"INCOMPLETO"` | i collegamenti ci sono, ma qualche campo manca nei file | tutte le variabili, quelle mancanti valgono `None` e sono elencate in `campi_mancanti` |
| `"NON_TROVATO"` | il sinistro non esiste, oppure manca un collegamento (polizza, cliente o requisiti) | `messaggio` che dice **quale** informazione manca e `dato_mancante` (`"sinistro"`, `"polizza"`, `"cliente"` o `"requisiti"`) |
| `"ERRORE_LETTURA"` | un file non esiste o non è un JSON valido | `messaggio` con la descrizione dell'errore |

Restituire un dizionario con `status` invece di bloccare il programma riprende lo stile dei tool visti a lezione (`"status": "OK"` / `"NOT_FOUND"`): l'agente può leggere `status` e decidere il passo successivo (per esempio chiedere chiarimenti o passare la pratica a un operatore).

**Come funziona, passo per passo.**
1. **Caricamento**: legge i quattro file. Se `carica_dati` solleva un errore, lo intercetta con `try/except` e restituisce `ERRORE_LETTURA`.
2. **Ricerca del sinistro** con `cerca_sinistro`. Se è `None` → `NON_TROVATO` (dato mancante: sinistro).
3. **Dal sinistro alla polizza**: usa il `policy_id` del sinistro. Se il collegamento manca o la polizza non esiste → `NON_TROVATO` (polizza).
4. **Dalla polizza al cliente**: usa il `customer_id` della polizza. Se manca → `NON_TROVATO` (cliente).
5. **Dal tipo di sinistro ai requisiti**: usa il `loss_type`. Se manca → `NON_TROVATO` (requisiti).
6. **Raccolta**: con `copia_campi` copia nel dizionario `dati_pratica` i campi utili da ciascun record.
7. Se qualche campo mancava, lo `status` diventa `INCOMPLETO`, altrimenti `OK`.

La funzione **rilegge i file a ogni chiamata**: i file sono piccoli e così, se i dati vengono aggiornati, si usa sempre la versione più recente.

**Esempio.** `estrai_variabili_pratica("CL004")` restituisce un dizionario con, tra gli altri, `"loss_type": "theft"`, `"coverage": "collision"`, `"documents": ["photo"]`, `"required_documents": ["police_report", "photo"]` e `"automatic_review_max": 1000`. Sarà poi il codice degli altri componenti a notare che la copertura non corrisponde e che manca un documento.

In [10]:
def risultato_non_trovato(dato_mancante, messaggio):
    """Crea il dizionario da restituire quando un'informazione non viene trovata."""
    return {"status": "NON_TROVATO", "dato_mancante": dato_mancante, "messaggio": messaggio}


def estrai_variabili_pratica(id_sinistro, cartella=CARTELLA_DATI):
    """Raccoglie in un unico dizionario tutte le informazioni di una pratica.

    Parametri:
        id_sinistro: codice del sinistro, ad esempio "CL001".
        cartella: cartella dei file JSON (di default la cartella "data").

    Restituisce:
        Un dizionario con la chiave "status" ("OK", "INCOMPLETO", "NON_TROVATO",
        "ERRORE_LETTURA"), un "messaggio" e, se la pratica è stata trovata,
        le variabili della pratica con le chiavi originali dei JSON.
    """
    # Il codice deve essere un testo: se arriva altro (es. un numero) lo segnaliamo
    if not isinstance(id_sinistro, str):
        return risultato_non_trovato("sinistro", f"Il codice del sinistro deve essere un testo, ricevuto: {id_sinistro!r}.")
    id_sinistro = id_sinistro.strip().upper()

    # 1. Caricamento dei file: qui possono avvenire solo errori di LETTURA
    try:
        sinistri = carica_dati("claims.json", cartella)
        polizze = carica_dati("policies.json", cartella)
        clienti = carica_dati("customers.json", cartella)
        requisiti = carica_dati("requirements.json", cartella)
    except (FileNotFoundError, ValueError) as errore:
        return {"status": "ERRORE_LETTURA", "messaggio": str(errore)}

    # 2. Sinistro
    sinistro = cerca_sinistro(id_sinistro, sinistri)
    if sinistro is None:
        return risultato_non_trovato("sinistro", f"Il sinistro '{id_sinistro}' non esiste in claims.json.")

    # 3. Dal sinistro alla polizza (collegamento: policy_id)
    id_polizza = sinistro.get("policy_id")
    if id_polizza is None:
        return risultato_non_trovato("polizza", f"Il sinistro '{id_sinistro}' non indica nessuna polizza (manca 'policy_id').")
    polizza = cerca_polizza(id_polizza, polizze)
    if polizza is None:
        return risultato_non_trovato("polizza", f"La polizza '{id_polizza}' del sinistro '{id_sinistro}' non esiste in policies.json.")

    # 4. Dalla polizza al cliente (collegamento: customer_id)
    id_cliente = polizza.get("customer_id")
    if id_cliente is None:
        return risultato_non_trovato("cliente", f"La polizza '{id_polizza}' non indica nessun cliente (manca 'customer_id').")
    cliente = cerca_cliente(id_cliente, clienti)
    if cliente is None:
        return risultato_non_trovato("cliente", f"Il cliente '{id_cliente}' della polizza '{id_polizza}' non esiste in customers.json.")

    # 5. Dal tipo di sinistro ai requisiti (collegamento: loss_type)
    tipo_sinistro = sinistro.get("loss_type")
    if tipo_sinistro is None:
        return risultato_non_trovato("requisiti", f"Il sinistro '{id_sinistro}' non indica il tipo di sinistro (manca 'loss_type').")
    requisito = cerca_requisiti(tipo_sinistro, requisiti)
    if requisito is None:
        return risultato_non_trovato("requisiti", f"Non ci sono requisiti per il tipo di sinistro '{tipo_sinistro}' in requirements.json.")

    # 6. Raccolta delle variabili, con le chiavi originali dei JSON
    dati_pratica = {}
    campi_mancanti = []
    copia_campi(sinistro, ["claim_id", "policy_id"], dati_pratica, campi_mancanti)
    copia_campi(polizza, ["customer_id"], dati_pratica, campi_mancanti)
    copia_campi(cliente, ["name", "risk_tier"], dati_pratica, campi_mancanti)
    copia_campi(polizza, ["product", "coverage", "active", "deductible", "max_claim"], dati_pratica, campi_mancanti)
    copia_campi(sinistro, ["loss_type", "estimated_damage", "documents"], dati_pratica, campi_mancanti)
    copia_campi(requisito, ["required_documents", "automatic_review_max"], dati_pratica, campi_mancanti)

    # 7. Esito del recupero (NON è la decisione sulla pratica)
    dati_pratica["campi_mancanti"] = campi_mancanti
    if len(campi_mancanti) > 0:
        dati_pratica["status"] = "INCOMPLETO"
        dati_pratica["messaggio"] = "Dati recuperati, ma mancano questi campi: " + ", ".join(campi_mancanti)
    else:
        dati_pratica["status"] = "OK"
        dati_pratica["messaggio"] = "Tutti i dati della pratica sono stati recuperati."
    return dati_pratica

La piccola funzione `risultato_non_trovato` serve solo a non ripetere quattro volte lo stesso dizionario di errore: riceve quale dato manca e il messaggio, e restituisce il dizionario con `status = "NON_TROVATO"`.

`isinstance(id_sinistro, str)` controlla che il codice sia un testo (`str`): se arrivasse un numero, `.strip()` causerebbe un errore.

## Esempi: estrazione delle pratiche CL001–CL004

La funzione `mostra_pratica` stampa il dizionario una riga per chiave, mostrando anche il **tipo** di ogni valore (`int` = numero intero, `bool` = vero/falso, `list` = lista, `str` = testo), così si vede che i tipi originali sono stati mantenuti.

Questi esempi verificano **solo il recupero dei dati**: non approvano e non respingono le pratiche.

In [11]:
def mostra_pratica(dati_pratica):
    """Stampa le variabili della pratica, una per riga, con il loro tipo Python."""
    for chiave, valore in dati_pratica.items():
        print(f"  {chiave:22} = {valore!r:40} ({type(valore).__name__})")


for codice in ["CL001", "CL002", "CL003", "CL004"]:
    print(f"=== Pratica {codice} ===")
    pratica = estrai_variabili_pratica(codice)
    mostra_pratica(pratica)
    print()

=== Pratica CL001 ===
  claim_id               = 'CL001'                                  (str)
  policy_id              = 'POL001'                                 (str)
  customer_id            = 'AS001'                                  (str)
  name                   = 'Pietro Sala'                            (str)
  risk_tier              = 'standard'                               (str)
  product                = 'Auto'                                   (str)
  coverage               = 'collision'                              (str)
  active                 = True                                     (bool)
  deductible             = 250                                      (int)
  max_claim              = 15000                                    (int)
  loss_type              = 'collision'                              (str)
  estimated_damage       = 1200                                     (int)
  documents              = ['photo', 'police_report']               (list)
  required_doc

### Cosa notare negli esempi (osservazioni, non decisioni)

I dati recuperati mostrano già le situazioni che gli altri componenti dovranno valutare con le loro regole:
- **CL002**: `documents` contiene solo `photo`, mentre `required_documents` chiede anche `repair_invoice`; `estimated_damage` (7000) è sopra `automatic_review_max` (5000).
- **CL003**: la polizza `POL003` ha `active = False`.
- **CL004**: `loss_type` è `theft` ma la polizza copre `collision`; manca `police_report`.

Il dizionario serve proprio a rendere questi confronti semplici per il codice successivo.

## Gestione degli errori

### Pratica inesistente: CL999
Il sinistro `CL999` non esiste. La funzione non blocca il notebook: restituisce `status = "NON_TROVATO"` e dice quale dato manca.

In [12]:
risultato = estrai_variabili_pratica("CL999")
mostra_pratica(risultato)

if risultato["status"] == "NON_TROVATO":
    print("\nInformazione mancante:", risultato["dato_mancante"])

  status                 = 'NON_TROVATO'                            (str)
  dato_mancante          = 'sinistro'                               (str)
  messaggio              = "Il sinistro 'CL999' non esiste in claims.json." (str)

Informazione mancante: sinistro


### Codice scritto in modo impreciso
Spazi e lettere minuscole vengono corretti prima della ricerca; un valore che non è testo viene segnalato.

In [13]:
print(estrai_variabili_pratica("  cl001 ")["status"])
print(estrai_variabili_pratica(1)["messaggio"])

OK
Il codice del sinistro deve essere un testo, ricevuto: 1.


### Errori di lettura: file mancante e JSON non valido

Per provare questi casi **senza toccare i file veri** del progetto, creiamo una cartella temporanea con `tempfile.TemporaryDirectory()` (libreria standard): viene cancellata da sola alla fine del blocco `with`.

La funzione `crea_cartella_test` copia nella cartella temporanea i dati veri e permette di sostituirne uno con un contenuto diverso. Così possiamo simulare dati rotti o collegamenti mancanti.

Notate la differenza: un **file mancante** o **rotto** dà `ERRORE_LETTURA`, un **elemento non trovato** dà `NON_TROVATO`.

In [14]:
import tempfile

def crea_cartella_test(cartella_test, file_modificati):
    """Copia i file JSON veri in 'cartella_test' e sostituisce quelli indicati.

    file_modificati: dizionario {nome_file: testo da scrivere al posto dell'originale}.
    """
    for nome_file in ["claims.json", "policies.json", "customers.json", "requirements.json"]:
        testo = (CARTELLA_DATI / nome_file).read_text(encoding="utf-8")
        if nome_file in file_modificati:
            testo = file_modificati[nome_file]
        (Path(cartella_test) / nome_file).write_text(testo, encoding="utf-8")


# a) File inesistente: chiamata diretta a carica_dati, errore intercettato
try:
    carica_dati("file_che_non_esiste.json")
except FileNotFoundError as errore:
    print("a) Errore gestito:", errore)

# b) JSON non valido (manca la parentesi di chiusura)
with tempfile.TemporaryDirectory() as cartella_test:
    crea_cartella_test(cartella_test, {"policies.json": '[{"policy_id": "POL001"'})
    risultato = estrai_variabili_pratica("CL001", cartella_test)
    print("b)", risultato["status"], "->", risultato["messaggio"])

a) Errore gestito: Il file 'data\file_che_non_esiste.json' non esiste. Controlla di aver avviato il notebook dalla cartella del progetto.
b) ERRORE_LETTURA -> Il file 'C:\Users\A847AP~1\AppData\Local\Temp\tmpi_jn8_w6\policies.json' non contiene un JSON valido (riga 1): Expecting ',' delimiter


### Collegamenti assenti e campi mancanti

Ora simuliamo dati con problemi nei collegamenti:
- **c)** il sinistro punta a una polizza che non esiste (`POL999`);
- **d)** la polizza punta a un cliente che non esiste;
- **e)** il tipo di sinistro non ha requisiti in `requirements.json`;
- **f)** alla polizza manca la franchigia (`deductible`): il collegamento c'è, quindi lo stato è `INCOMPLETO` e la franchigia vale `None`, **non** `0`.

In [15]:
casi_di_prova = {
    "c) polizza inesistente": {"claims.json": '[{"claim_id": "CL001", "policy_id": "POL999", "loss_type": "collision", "estimated_damage": 1200, "documents": ["photo"]}]'},
    "d) cliente inesistente": {"policies.json": '[{"policy_id": "POL001", "customer_id": "AS999", "product": "Auto", "coverage": "collision", "deductible": 250, "max_claim": 15000, "active": true}]'},
    "e) requisiti mancanti": {"requirements.json": '[]'},
    "f) franchigia mancante": {"policies.json": '[{"policy_id": "POL001", "customer_id": "AS001", "product": "Auto", "coverage": "collision", "max_claim": 15000, "active": true}]'},
}

for descrizione, file_modificati in casi_di_prova.items():
    with tempfile.TemporaryDirectory() as cartella_test:
        crea_cartella_test(cartella_test, file_modificati)
        risultato = estrai_variabili_pratica("CL001", cartella_test)
    print(f"{descrizione}: {risultato['status']} -> {risultato['messaggio']}")
    if risultato["status"] == "INCOMPLETO":
        print("   deductible =", risultato["deductible"], "| campi_mancanti =", risultato["campi_mancanti"])

c) polizza inesistente: NON_TROVATO -> La polizza 'POL999' del sinistro 'CL001' non esiste in policies.json.
d) cliente inesistente: NON_TROVATO -> Il cliente 'AS999' della polizza 'POL001' non esiste in customers.json.
e) requisiti mancanti: NON_TROVATO -> Non ci sono requisiti per il tipo di sinistro 'collision' in requirements.json.
f) franchigia mancante: INCOMPLETO -> Dati recuperati, ma mancano questi campi: deductible
   deductible = None | campi_mancanti = ['deductible']


## Controlli automatici

Questa cella verifica con `assert` che i risultati siano quelli attesi. `assert condizione` non fa nulla se la condizione è vera; se è falsa blocca la cella con un errore. Se la cella stampa il messaggio finale, tutti i controlli sono passati.

In [16]:
# Valori attesi letti a mano dai file JSON del progetto
pratica = estrai_variabili_pratica("CL002")
assert pratica["status"] == "OK"
assert pratica["policy_id"] == "POL002" and pratica["customer_id"] == "AS002"
assert pratica["name"] == "Chiara Riva" and pratica["risk_tier"] == "premium"
assert pratica["estimated_damage"] == 7000 and type(pratica["estimated_damage"]) is int
assert pratica["documents"] == ["photo"]
assert pratica["required_documents"] == ["photo", "repair_invoice"]
assert pratica["automatic_review_max"] == 5000

pratica = estrai_variabili_pratica("CL003")
assert pratica["active"] is False          # booleano originale, non il testo "false"

pratica = estrai_variabili_pratica("CL004")
assert pratica["policy_id"] == "POL001" and pratica["name"] == "Pietro Sala"
assert pratica["loss_type"] == "theft" and pratica["coverage"] == "collision"

# Tutte le pratiche esistenti hanno tutte le chiavi richieste
chiavi_attese = ["claim_id", "policy_id", "customer_id", "name", "risk_tier", "product",
                 "coverage", "active", "deductible", "max_claim", "loss_type",
                 "estimated_damage", "documents", "required_documents", "automatic_review_max"]
for codice in ["CL001", "CL002", "CL003", "CL004"]:
    pratica = estrai_variabili_pratica(codice)
    assert pratica["status"] == "OK", codice
    for chiave in chiavi_attese:
        assert chiave in pratica, f"{codice}: manca {chiave}"

# Pratica inesistente
assert estrai_variabili_pratica("CL999")["status"] == "NON_TROVATO"
assert estrai_variabili_pratica("CL999")["dato_mancante"] == "sinistro"

print("Tutti i controlli sono passati.")

Tutti i controlli sono passati.


## Come usare `estrai_variabili_pratica` negli altri componenti

1. Chiamare la funzione e **controllare per prima cosa `status`**:
   - `"OK"` → si possono applicare le regole;
   - `"INCOMPLETO"` → leggere `campi_mancanti` (es. fase di chiarimento o intervento umano);
   - `"NON_TROVATO"` → leggere `dato_mancante` e `messaggio` (es. chiedere all'utente il codice corretto);
   - `"ERRORE_LETTURA"` → errore del tool: gestirlo come *tool failure*.
2. Leggere le variabili con le chiavi originali, ad esempio:

```python
pratica = estrai_variabili_pratica("CL002")
if pratica["status"] == "OK":
    danno = pratica["estimated_damage"]          # numero
    soglia = pratica["automatic_review_max"]     # numero
    polizza_attiva = pratica["active"]           # booleano
    documenti_presenti = pratica["documents"]    # lista
    documenti_richiesti = pratica["required_documents"]  # lista
    # ...qui le regole deterministiche degli altri componenti
```

3. In LangGraph il dizionario si può salvare nello **stato condiviso** (ad esempio in un campo `dati_pratica`), e la funzione si può trasformare in tool aggiungendo `@tool` di LangChain, come nei lab del II anno.